In [1]:
import pandas as pd
import os
from PIL import Image
import torchvision.transforms as transforms
from transformers import Blip2Processor
import re
from sklearn.model_selection import train_test_split
from torch.utils.data import Dataset
import torch

# 1. Paths to folders and files
images_folder = 'Images'  # Local folder containing images
outer_captions_folder = 'captions.txt'  # Outer folder

# 2. Check if folders exist
if not os.path.exists(images_folder):
    raise FileNotFoundError(f"Images folder not found at {images_folder}")

if not os.path.isdir(outer_captions_folder):
    raise FileNotFoundError(f"Outer captions folder not found at {outer_captions_folder}")

# Path to the inner captions folder
inner_captions_folder = os.path.join(outer_captions_folder, 'captions.txt')

if not os.path.isdir(inner_captions_folder):
    raise FileNotFoundError(f"Inner captions folder not found at {inner_captions_folder}")

# Path to the actual captions.txt file
captions_txt_file = os.path.join(inner_captions_folder, 'captions.txt')

if not os.path.isfile(captions_txt_file):
    raise FileNotFoundError(f"Captions text file not found at {captions_txt_file}")

print("✅ Images folder and captions file found.")

# 3. Load the captions file
captions = pd.read_csv(captions_txt_file, delimiter=',', names=['image', 'caption'], header=None)

# 4. Sample images to ensure exactly 5 captions per image, totaling 8000 image-caption pairs
captions_per_image = 5
num_images_to_sample = 8000 // captions_per_image  # 1600 images

# Filter for images with exactly 5 captions
image_caption_counts = captions.groupby('image').size()
valid_images = image_caption_counts[image_caption_counts == captions_per_image].index
captions = captions[captions['image'].isin(valid_images)].reset_index(drop=True)

# Verify we have enough valid images
unique_images = captions['image'].unique()
if len(unique_images) < num_images_to_sample:
    raise ValueError(f"Not enough images with exactly 5 captions to sample. Need {num_images_to_sample}, but only {len(unique_images)} available.")

# Sample 1600 unique images
sampled_images = pd.Series(unique_images).sample(n=num_images_to_sample, random_state=42).tolist()

# Filter captions for the sampled images
sampled_captions = captions[captions['image'].isin(sampled_images)].reset_index(drop=True)

# Verify we have exactly 8000 pairs
if len(sampled_captions) != 8000:
    raise ValueError(f"Expected 8000 image-caption pairs, but got {len(sampled_captions)}.")

# Save to CSV
sampled_captions.to_csv('subsampled_captions.csv', index=False)
print(f"✅ Sampled {len(sampled_captions)} image-caption pairs, with {len(sampled_captions['image'].unique())} unique images, each having exactly 5 captions.")

# 5. Image transformation
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# 6. Load and preprocess images
def load_image(image_name):
    image_path = os.path.join(images_folder, image_name.split('#')[0])
    try:
        image = Image.open(image_path).convert('RGB')
        return transform(image)
    except Exception as e:
        print(f"❌ Error loading {image_path}: {e}")
        return None

# 7. Preprocess and tokenize captions
processor = Blip2Processor.from_pretrained('Salesforce/blip2-flan-t5-xl')

def preprocess_caption(caption):
    caption = caption.lower()
    caption = re.sub(r'[^\w\s]', '', caption)
    return caption

def tokenize_caption(caption):
    # Tokenize the caption for decoder input (used as labels during training)
    tokens = processor.tokenizer(
        caption,
        padding='max_length',
        max_length=32,
        truncation=True,
        return_tensors='pt'
    )
    return tokens

# Create a minimal input_ids with just the start token
start_token_id = processor.tokenizer.bos_token_id
if start_token_id is None:
    # Fallback to the ID of the <s> token (common BOS token for T5-based models)
    start_token_id = processor.tokenizer.convert_tokens_to_ids('<s>')
    if start_token_id is None:
        raise ValueError("Could not determine a start token ID. Ensure the tokenizer has a BOS token or '<s>' token.")
input_ids = torch.tensor([start_token_id], dtype=torch.long)  # Shape: (1,)
attention_mask = torch.tensor([1], dtype=torch.long)  # Shape: (1,)

sampled_captions['cleaned_caption'] = sampled_captions['caption'].apply(preprocess_caption)
sampled_captions['tokens'] = sampled_captions['cleaned_caption'].apply(tokenize_caption)

# 8. Train, validation, test split
train_df, temp_df = train_test_split(sampled_captions, test_size=0.2, random_state=42)
val_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=42)

print(f"✅ Dataset sizes -> Train: {len(train_df)}, Val: {len(val_df)}, Test: {len(test_df)}")

# 9. PyTorch Dataset
class Flickr8kDataset(Dataset):
    def __init__(self, df, image_folder, transform):
        self.df = df
        self.image_folder = image_folder
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image = load_image(row['image'])
        tokens = row['tokens']
        if image is None:
            return None
        # Prepare decoder_input_ids and labels for training
        decoder_input_ids = tokens['input_ids'].squeeze()[:-1]  # Shift by removing the last token
        labels = tokens['input_ids'].squeeze()[1:]  # Shift by removing the first token for loss computation
        decoder_attention_mask = tokens['attention_mask'].squeeze()[:-1]  # Match decoder_input_ids length
        return {
            'pixel_values': image,
            'input_ids': input_ids,  # Minimal input_ids (start token)
            'attention_mask': attention_mask,  # Minimal attention_mask
            'decoder_input_ids': decoder_input_ids,
            'decoder_attention_mask': decoder_attention_mask,
            'labels': labels
        }

# 10. Initialize datasets
train_dataset = Flickr8kDataset(train_df, images_folder, transform)
val_dataset = Flickr8kDataset(val_df, images_folder, transform)
test_dataset = Flickr8kDataset(test_df, images_folder, transform)

print("✅ Datasets initialized successfully!")

✅ Images folder and captions file found.


Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


✅ Sampled 8000 image-caption pairs, with 1600 unique images, each having exactly 5 captions.
✅ Dataset sizes -> Train: 6400, Val: 800, Test: 800
✅ Datasets initialized successfully!


In [ ]:
from transformers import Blip2ForConditionalGeneration, Blip2Processor, Trainer, TrainingArguments
import torch
from PIL import Image
import os

# --- Model & Training Pipeline: Load BLIP-2, Apply Fine-Tuning Strategies ---
# 1. Load the BLIP-2 model and processor
try:
    model = Blip2ForConditionalGeneration.from_pretrained(
        'Salesforce/blip2-flan-t5-xl',
        torch_dtype=torch.float16,
        device_map="auto"
    )
    processor = Blip2Processor.from_pretrained('Salesforce/blip2-flan-t5-xl')
    print("✅ Model and processor loaded successfully!")
except Exception as e:
    print(f"❌ Error loading model: {e}")
    raise

# 2. Define fine-tuning strategies
def setup_finetuning(strategy='full'):
    if strategy == 'layer_freeze':
        for param in model.vision_model.parameters():
            param.requires_grad = False
        print("✅ Vision model parameters frozen.")
        return model
    else:
        print("✅ Full fine-tuning enabled.")
        return model

# Choose fine-tuning strategy
model = setup_finetuning(strategy='layer_freeze')

# 3. Assume train_dataset and val_dataset are already defined
batch_size = 8
steps_per_epoch = len(train_dataset) // batch_size
if len(train_dataset) % batch_size != 0:
    steps_per_epoch += 1

# 4. Define training arguments (fixed version)
training_args = TrainingArguments(
    output_dir='./blip2_finetuned',
    num_train_epochs=3,
    per_device_train_batch_size=batch_size,
    per_device_eval_batch_size=batch_size,
    eval_steps=steps_per_epoch,  # Evaluate every epoch
    save_steps=steps_per_epoch,
    logging_dir='./logs',
    learning_rate=5e-5,
    fp16=True,
    report_to="none"
)

# 5. Initialize Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset
)

# 6. Train the model
print("🚀 Starting model training...")
trainer.train()
print("✅ Training completed!")

# --- Generative Decoding: Implement Beam Search, Top-k, Top-p Sampling, Temperature Control ---
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.eval()  # Removed model.to(device)

# Caption generation function
def generate_caption(image, method='beam', **kwargs):
    inputs = processor(images=image, return_tensors='pt').to(device)
    if method == 'beam':
        outputs = model.generate(
            **inputs,
            max_length=32,
            num_beams=kwargs.get('num_beams', 5),
            early_stopping=True
        )
    elif method == 'top_k':
        outputs = model.generate(
            **inputs,
            max_length=32,
            do_sample=True,
            top_k=kwargs.get('top_k', 50)
        )
    elif method == 'top_p':
        outputs = model.generate(
            **inputs,
            max_length=32,
            do_sample=True,
            top_p=kwargs.get('top_p', 0.9),
            temperature=kwargs.get('temperature', 1.0)
        )
    return processor.decode(outputs[0], skip_special_tokens=True)

# Example: Generate captions with different methods
# Load the raw image for inference (not preprocessed tensor)
image_path = os.path.join(images_folder, sampled_captions['image'].iloc[0].split('#')[0])
sample_image = Image.open(image_path).convert('RGB')  # Load as PIL image
print("Beam Search:", generate_caption(sample_image, method='beam', num_beams=5))
print("Top-k Sampling:", generate_caption(sample_image, method='top_k', top_k=50))
print("Top-p Sampling:", generate_caption(sample_image, method='top_p', top_p=0.9, temperature=0.7))

# 7. Save the fine-tuned model and processor
model.save_pretrained("./fintuned_images")
processor.save_pretrained("./fintuned_images")
print("✅ Model and processor saved to ./fintuned_images")

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

✅ Model and processor loaded successfully!
✅ Vision model parameters frozen.
🚀 Starting model training...


Passing a tuple of `past_key_values` is deprecated and will be removed in Transformers v4.48.0. You should pass an instance of `EncoderDecoderCache` instead, e.g. `past_key_values=EncoderDecoderCache.from_legacy_cache(past_key_values)`.


In [ ]:

# --- Evaluation Metrics: BLEU-4, METEOR, ROUGE-L, SPICE, Self-BLEU, Distinct-n ---
# Required imports for evaluation metrics
import nltk
from nltk.translate.bleu_score import corpus_bleu, SmoothingFunction
from nltk.tokenize import word_tokenize
from rouge_score import rouge_scorer
from collections import Counter
import numpy as np

# Download NLTK data
nltk.download('punkt')
nltk.download('wordnet')

# Note: pycocoevalcap requires installation and additional setup for METEOR and SPICE.
# For simplicity, we'll implement BLEU-4, ROUGE-L, Self-BLEU, and Distinct-n here.
# METEOR and SPICE require external dependencies (pycocoevalcap, Java for METEOR, etc.).
# Users can install pycocoevalcap via: pip install pycocoevalcap
# However, we'll provide placeholders for METEOR and SPICE.

try:
    from pycocoevalcap.meteor.meteor import Meteor
    from pycocoevalcap.spice.spice import Spice
    METEOR_AVAILABLE = True
    SPICE_AVAILABLE = True
except ImportError:
    METEOR_AVAILABLE = False
    SPICE_AVAILABLE = False
    print("⚠️ METEOR and SPICE evaluation requires pycocoevalcap. Install it using: pip install pycocoevalcap")
    print("METEOR also requires Java to be installed on your system.")

# Load test dataset captions (ground truth)
# Since test_dataset is a Flickr8kDataset, we can access the underlying DataFrame
test_captions_df = test_dataset.df  # Access the DataFrame used to create test_dataset
ground_truth_captions = test_captions_df.groupby('image')['cleaned_caption'].apply(list).to_dict()

# Generate captions for all test images using Beam Search
generated_captions = []
image_names = test_captions_df['image'].unique()

print("Generating captions for test images...")
for img_name in image_names:
    image_path = os.path.join(images_folder, img_name.split('#')[0])
    try:
        image = Image.open(image_path).convert('RGB')
        caption = generate_caption(image, method='beam', num_beams=5)
        generated_captions.append(caption)
    except Exception as e:
        print(f"❌ Error generating caption for {img_name}: {e}")
        generated_captions.append("")  # Append empty caption to maintain alignment

# Create a mapping of image names to generated captions
generated_captions_dict = dict(zip(image_names, generated_captions))

# Prepare references and hypotheses for evaluation
references = []  # List of lists: [[ref1, ref2, ...], [ref1, ref2, ...], ...]
hypotheses = []  # List of generated captions

for img_name in image_names:
    refs = ground_truth_captions.get(img_name, [])
    hyp = generated_captions_dict.get(img_name, "")
    if hyp:  # Only include images where a caption was successfully generated
        # Tokenize references and hypothesis for BLEU and other metrics
        refs_tokenized = [word_tokenize(ref) for ref in refs]
        hyp_tokenized = word_tokenize(hyp)
        references.append(refs_tokenized)
        hypotheses.append(hyp_tokenized)

# 1. BLEU-4
# Compute corpus-level BLEU-4 with smoothing
smoothing = SmoothingFunction().method1
bleu4_score = corpus_bleu(references, hypotheses, weights=(0.25, 0.25, 0.25, 0.25), smoothing_function=smoothing)
print(f"BLEU-4 Score: {bleu4_score:.4f}")

# 2. METEOR (if available)
if METEOR_AVAILABLE:
    meteor_scorer = Meteor()
    # Prepare data in the format required by pycocoevalcap: {img_id: [caption]}
    gts = {i: [" ".join(ref) for ref in refs] for i, refs in enumerate(references)}
    res = {i: [" ".join(hyp)] for i, hyp in enumerate(hypotheses)}
    meteor_score, _ = meteor_scorer.compute_score(gts, res)
    print(f"METEOR Score: {meteor_score:.4f}")
else:
    print("METEOR Score: Not computed (pycocoevalcap not installed).")

# 3. ROUGE-L
rouge_scorer_obj = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=True)
rouge_l_scores = []
for refs, hyp in zip(references, hypotheses):
    ref_strs = [" ".join(ref) for ref in refs]
    hyp_str = " ".join(hyp)
    # Compute ROUGE-L for each reference and take the maximum
    scores = [rouge_scorer_obj.score(ref_str, hyp_str)['rougeL'].fmeasure for ref_str in ref_strs]
    rouge_l_scores.append(max(scores))
rouge_l_score = np.mean(rouge_l_scores)
print(f"ROUGE-L Score: {rouge_l_score:.4f}")

# 4. SPICE (if available)
if SPICE_AVAILABLE:
    spice_scorer = Spice()
    gts = {i: [" ".join(ref) for ref in refs] for i, refs in enumerate(references)}
    res = {i: [" ".join(hyp)] for i, hyp in enumerate(hypotheses)}
    spice_score, _ = spice_scorer.compute_score(gts, res)
    print(f"SPICE Score: {spice_score:.4f}")
else:
    print("SPICE Score: Not computed (pycocoevalcap not installed).")

# 5. Self-BLEU (measures diversity by computing BLEU between generated captions)
self_bleu_scores = []
for i in range(len(hypotheses)):
    hyp_i = hypotheses[i]
    other_hyps = [hypotheses[j] for j in range(len(hypotheses)) if j != i]
    if other_hyps:  # Ensure there are other hypotheses to compare against
        self_bleu = corpus_bleu([other_hyps], [hyp_i], weights=(0.25, 0.25, 0.25, 0.25), smoothing_function=smoothing)
        self_bleu_scores.append(self_bleu)
self_bleu_score = np.mean(self_bleu_scores) if self_bleu_scores else 0.0
print(f"Self-BLEU Score: {self_bleu_score:.4f} (lower is better for diversity)")

# 6. Distinct-n (n=1, 2 for unigrams and bigrams)
def compute_distinct_n(captions, n):
    all_ngrams = []
    for caption in captions:
        # Generate n-grams
        ngrams = [tuple(caption[i:i+n]) for i in range(len(caption) - n + 1)]
        all_ngrams.extend(ngrams)
    if not all_ngrams:
        return 0.0
    distinct_ngrams = len(set(all_ngrams))
    total_ngrams = len(all_ngrams)
    return distinct_ngrams / total_ngrams

distinct_1 = compute_distinct_n(hypotheses, 1)
distinct_2 = compute_distinct_n(hypotheses, 2)
print(f"Distinct-1 Score: {distinct_1:.4f} (higher is better for diversity)")
print(f"Distinct-2 Score: {distinct_2:.4f} (higher is better for diversity)")